# 🍏 Basic Retrieval-Augmented Generation (RAG) with a Foundry Agent 🍎

In this notebook, we'll demonstrate a RAG flow using:

- `azure-ai-projects` (endpoint-based `AIProjectClient`)
- A keyless Microsoft Foundry client for embeddings
- `azure-search-documents` (to build the vector index)
- A **Microsoft Foundry Agent** with the built-in **Azure AI Search Service (Foundry IQ) tool** to handle retrieval and answer generation

Our theme is Health & Fitness 🍏 so we'll create a simple set of health tips, embed them, store them in a search index, then create an agent that can search that index and answer questions grounded in it — with citations.

**Disclaimer:** This is not medical advice. For real health questions, consult a professional.

## What is RAG?
Retrieval-Augmented Generation (RAG) is a technique where the LLM uses relevant retrieved text chunks from your data to craft a final answer. This helps ground the model's response in real data, reducing hallucinations.

## What changed from the previous version of this notebook
Previously, this notebook manually embedded the user's query, ran a `VectorizedQuery` against the index, stuffed the results into a system prompt, and called Chat Completions directly. That entire manual pipeline is now replaced by a **Foundry Agent with the Azure AI Search tool** — the agent handles embedding the query, searching the index, and generating a grounded, cited answer internally. You just create the agent once and ask it questions.

## 1. Setup

We'll import libraries, load environment variables, create an `AIProjectClient`, and build a keyless Azure OpenAI client (used for embeddings only now — the agent handles chat/generation).

This needs `PROJECT_ENDPOINT`, `AZURE_OPENAI_ENDPOINT`, `SEARCH_ENDPOINT`, `SEARCH_API_KEY`, and `SEARCH_CONNECTION_NAME` in your `.env`.

> **New requirement:** `SEARCH_CONNECTION_NAME` — the name of the connection between your Foundry project and your Azure AI Search resource. Create this once in the Foundry portal under **Operate > Admin > Add connection** if you haven't already (see [Add a new connection to your project](https://learn.microsoft.com/en-us/azure/foundry/how-to/connections-add)).

> **Complete `2-embeddings.ipynb` notebook before starting this one**

In [ ]:
import os
import re
from dotenv import load_dotenv
from pathlib import Path

# azure-ai-projects (endpoint-based client)
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

# For creating the vector index
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

chat_model = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4")
embedding_model = os.environ.get("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-3-large")
search_index_name = os.environ.get("SEARCH_INDEX_NAME", "healthtips-index")
search_connection_name = os.environ["SEARCH_CONNECTION_NAME"]

# Azure AI Search endpoint + admin key (from the Search resource in the Azure portal)
# Used here only to build the index. The agent itself reaches Search through the
# project connection above, not through this key.
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_API_KEY"]

try:
    # Project client confirms the Foundry project is reachable, and gives us
    # both the OpenAI-compatible client and the Agents client.
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()

    # Separate keyless (Microsoft Entra ID) Azure OpenAI client, used only for
    # embeddings, since embeddings are served from the Azure OpenAI endpoint
    # rather than the project's /openai/v1 route.
    aoai_endpoint = re.match(r"https://[^/]+", os.environ["AZURE_OPENAI_ENDPOINT"]).group(0)
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
    )
    embedding_client = AzureOpenAI(
        azure_endpoint=aoai_endpoint,
        azure_ad_token_provider=token_provider,
        api_version=os.environ.get("OPENAI_API_VERSION", "2024-10-21"),
    )
    print("✅ AIProjectClient + Azure OpenAI (embeddings) client created successfully!")
except Exception as e:
    print("❌ Error creating clients:", e)

## 2. Create Sample Health Data

We'll create a few short doc chunks. In a real scenario, you might read from CSV or PDFs, chunk them up, embed them, and store them in your search index.

In [ ]:
health_tips = [
    {"id": "doc1", "content": "Daily 30-minute walks help maintain a healthy weight and reduce stress.", "source": "General Fitness"},
    {"id": "doc2", "content": "Stay hydrated by drinking 8-10 cups of water per day.", "source": "General Fitness"},
    {"id": "doc3", "content": "Consistent sleep patterns (7-9 hours) improve muscle recovery.", "source": "General Fitness"},
    {"id": "doc4", "content": "For cardio endurance, try interval training like HIIT.", "source": "Workout Advice"},
    {"id": "doc5", "content": "Warm up with dynamic stretches before running to reduce injury risk.", "source": "Workout Advice"},
    {"id": "doc6", "content": "Balanced diets typically include protein, whole grains, fruits, vegetables, and healthy fats.", "source": "Nutrition"},
]
print("Created a small list of health tips.")

## 3. Create the Vector Index and Upload Health Tips 🏋️

This part is unchanged from before: the Azure AI Search Service (Foundry IQ) tool queries an **existing** index, it doesn't create one for you. So we still need to build the index and load our health tips into it.

This creates our knowledge base — the 'fitness library' our agent will search through. 📚💪

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
)

def create_healthtips_index(endpoint: str, api_key: str, index_name: str, aoai_endpoint: str, embedding_deployment: str, dimension: int = 3072):
    """Create or update a search index for health tips with vector search capability.

    Includes an AzureOpenAIVectorizer so the Azure AI Search tool (used by our Foundry
    agent) can embed a natural-language query at query time. Without this, query types
    like 'vector' or 'vector_semantic_hybrid' fail with:
    'Query type ... requires a vector field with integrated vectorizer, but none was found'
    """
    index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

    try:
        index_client.delete_index(index_name)
        print(f"Deleted existing index: {index_name}")
    except Exception:
        pass  # Index doesn't exist yet

    vectorizer_name = "healthtips-openai-vectorizer"

    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(m=4, ef_construction=400, ef_search=500, metric=VectorSearchAlgorithmMetric.COSINE),
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="myHnswProfile",
                algorithm_configuration_name="myHnsw",
                vectorizer_name=vectorizer_name,
            )
        ],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name=vectorizer_name,
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=aoai_endpoint,
                    deployment_name=embedding_deployment,
                    model_name="text-embedding-3-large",
                    # No api_key/auth_identity set -> uses the search service's
                    # system-assigned managed identity (keyless). That identity
                    # needs the 'Cognitive Services OpenAI User' role on the
                    # Azure OpenAI / Foundry resource.
                ),
            )
        ],
    )

    # NOTE: the Azure AI Search tool needs a retrievable text field to cite ("content"),
    # and ideally a source URL field for clickable citations. We keep "source" as a
    # plain label here for simplicity; swap it for a real URL field if you want live links.
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String),
        SearchField(
            name="embedding",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=dimension,
            vector_search_profile_name="myHnswProfile",
        ),
    ]
    index_def = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search,
    semantic_search=SemanticSearch(
        default_configuration_name="healthtips-semantic-config",
        configurations=[
            SemanticConfiguration(
                name="healthtips-semantic-config",
                prioritized_fields=SemanticPrioritizedFields(
                    content_fields=[SemanticField(field_name="content")],
                ),
            )
        ],
    ),
)
    index_client.create_index(index_def)
    print(f"✅ Created or reset index: {index_name}")

## 3.1. Create Index & Upload Health Tips

In [ ]:
# Step 1: Confirm the embedding dimension using a sample document
sample_doc = health_tips[0]
emb_response = embedding_client.embeddings.create(model=embedding_model, input=[sample_doc["content"]])
embedding_length = len(emb_response.data[0].embedding)
print(f"✅ Got embedding length: {embedding_length}")

# Step 2: Create the index (now includes an integrated vectorizer)
create_healthtips_index(
    endpoint=search_endpoint,
    api_key=search_api_key,
    index_name=search_index_name,
    aoai_endpoint=aoai_endpoint,
    embedding_deployment=embedding_model,
    dimension=embedding_length,
)

# Step 3: Create search client for uploading documents
search_client = SearchClient(endpoint=search_endpoint, index_name=search_index_name, credential=AzureKeyCredential(search_api_key))

# Step 4: Embed and upload documents
search_docs = []
for doc in health_tips:
    emb_response = embedding_client.embeddings.create(model=embedding_model, input=[doc["content"]])
    search_docs.append({
        "id": doc["id"],
        "content": doc["content"],
        "source": doc["source"],
        "embedding": emb_response.data[0].embedding,
    })

search_client.upload_documents(documents=search_docs)
print(f"✅ Uploaded {len(search_docs)} documents to search index '{search_index_name}'")

# Step 1: Confirm the embedding dimension using a sample document
sample_doc = health_tips[0]
emb_response = embedding_client.embeddings.create(model=embedding_model, input=[sample_doc["content"]])
embedding_length = len(emb_response.data[0].embedding)
print(f"✅ Got embedding length: {embedding_length}")

# Step 2: Create the index (now includes an integrated vectorizer)
create_healthtips_index(
    endpoint=search_endpoint,
    api_key=search_api_key,
    index_name=search_index_name,
    aoai_endpoint=aoai_endpoint,
    embedding_deployment=embedding_model,
    dimension=embedding_length,
)

# Step 3: Create search client for uploading documents
search_client = SearchClient(endpoint=search_endpoint, index_name=search_index_name, credential=AzureKeyCredential(search_api_key))

# Step 4: Embed and upload documents
search_docs = []
for doc in health_tips:
    emb_response = embedding_client.embeddings.create(model=embedding_model, input=[doc["content"]])
    search_docs.append({
        "id": doc["id"],
        "content": doc["content"],
        "source": doc["source"],
        "embedding": emb_response.data[0].embedding,
    })

search_client.upload_documents(documents=search_docs)
print(f"✅ Uploaded {len(search_docs)} documents to search index '{search_index_name}'")

## 4. Create a Foundry Agent with the Azure AI Search Service (Foundry IQ) Tool 🤖

This is the section that replaces the old manual RAG pipeline (query embedding + `VectorizedQuery` + prompt-stuffing + Chat Completions).

Instead, we create **one agent**, attach the **Azure AI Search Service (Foundry IQ) tool**, and point it at our index through a **project connection** (set up once in the Foundry portal — see the `.env` note in Section 1). The agent handles embedding the query, searching, and grounding its answer with citations, all internally.

Reference: [Connect an Azure AI Search index to Foundry agents](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/ai-search)

In [ ]:
from azure.ai.projects.models import (
    AzureAISearchTool,
    PromptAgentDefinition,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)

# Resolve the connection ID from the connection name (created once in the Foundry portal)
azs_connection = project_client.connections.get(search_connection_name)
connection_id = azs_connection.id

agent = project_client.agents.create_version(
    agent_name="HealthTipsAgent",
    definition=PromptAgentDefinition(
        model=chat_model,
        instructions=(
            "You are a health & fitness assistant. Answer user questions using the "
            "search tool, grounded only in the retrieved health tips. "
            "Always provide citations for answers using the tool and render them as: "
            "`[message_idx:search_idx†source]`. If you're unsure, say 'I'm not sure'."
        ),
        tools=[
            AzureAISearchTool(
                azure_ai_search=AzureAISearchToolResource(
                    indexes=[
                        AISearchIndexResource(
                            project_connection_id=connection_id,
                            index_name=search_index_name,
                            query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
                        ),
                    ]
                )
            )
        ],
    ),
    description="Health tips RAG agent backed by Azure AI Search.",
)
print(f"✅ Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

## 5. Ask the Agent a Question 🎉

Instead of calling `chat.completions.create()` with a manually built prompt, we call the **Responses API** and reference our agent directly. The agent takes care of retrieval and grounding.

In [ ]:
def rag_chat(query: str) -> str:
    response = openai_client.responses.create(
        input=query,
        tool_choice="required",
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    return response.output_text

user_query = "What's a good short cardio routine for me if I'm busy?"
answer = rag_chat(user_query)
print("🗣️ User Query:", user_query)
print("🤖 RAG Answer:", answer)

## 6. Clean Up

Agents persist in your project until deleted. Clean up the agent version when you're done experimenting.

In [ ]:
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("🗑️ Agent deleted")

## 7. Conclusion

We've demonstrated a RAG pipeline using:

- Embedding docs & storing them in Azure AI Search Service (Foundry IQ) (unchanged).
- A **Foundry Agent with the Azure AI Search Service (Foundry IQ) tool** for retrieval and grounded, cited answers — replacing the previous manual embed → vector search → prompt-stuff → Chat Completions pipeline.

🔎 For a managed knowledge base experience instead of pointing directly at one index, see [Foundry IQ](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/foundry-iq-connect).

🚀 Want to optimize this further with a small language model? Check out the next notebook to see how to use a small model with the same Microsoft Foundry SDK!